# Flow outputs and polarity

Flow mass is a queryable edge trace. Aggregate with an explicit `side=`
(`"source"` or `"dest"`), turn instantaneous rates into counts with
`.incidence()`, and compare the two sides of the same flow.

In [ ]:
import numpy as np

from summer4 import (
    Dest,
    Everything,
    ExitFlow,
    FlowMass,
    FlowModel,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    Source,
    TraitChain,
    TransitionFlow,
)

state = Property("state", ("S", "I", "R"))
age = Property("age", ("0-4", "5-9", "10+"))
pmap = PropertyMap.from_property(state).stratify(age)

model = FlowModel(pmap)
model.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["I"],
        0.3,
    )
)
model.add_flow(
    TransitionFlow(
        "ageing",
        age.present(),
        age.present(),
        0.2,
        pairing=TraitChain(age, (("0-4", "5-9"), ("5-9", "10+"))),
    )
)
model.add_flow(ExitFlow("death", state["I"], 0.05))
cm = model.compile()

y0 = PropertyData.wrap(pmap, np.ones(pmap.size))
y0 = y0.at[state["S"] & age["0-4"]].set(500.0)
y0 = y0.at[state["I"] & age["0-4"]].set(10.0)

plan = SavePlan(
    requests={
        "infection": SaveRequest(FlowMass(flow="infection")),
        "ageing": SaveRequest(FlowMass(flow="ageing")),
        "death": SaveRequest(FlowMass(flow="death")),
    }
)
res = cm.run({}, y0, t0=0.0, steps=28, dt=1.0, save=plan)
assert res["infection"].dims == ("time", "edge")


## Source vs dest on the same flow

Ageing moves people from one band to the next. Grouping by the **source** age
counts who left each band; grouping by the **dest** age counts who arrived.
Neither is a default — `side=` is required.

In [ ]:
by_src = res["ageing"].sum_over(age, side="source").total()
by_dest = res["ageing"].sum_over(age, side="dest").total()
src_frame = res["ageing"].sum_over(age, side="source").to_frame()
dest_frame = res["ageing"].sum_over(age, side="dest").to_frame()
assert list(src_frame.columns)[1:] == ["age=0-4", "age=5-9", "age=10+"]
assert list(dest_frame.columns)[1:] == ["age=0-4", "age=5-9", "age=10+"]
# Nobody ages *out* of 10+; nobody ages *into* 0-4.
assert float(np.asarray(by_src.values)[0]) > 0.0
src_at_0 = np.asarray(res["ageing"].sum_over(age, side="source").values.data)[0]
dest_at_0 = np.asarray(res["ageing"].sum_over(age, side="dest").values.data)[0]
assert float(src_at_0[2]) == 0.0
assert float(dest_at_0[0]) == 0.0
assert not np.allclose(src_at_0, dest_at_0)
print("source age masses at t0:", src_at_0)
print("dest age masses at t0:  ", dest_at_0)


## Weekly incidence by age of the newly infected

Saved flow mass is an **instantaneous rate**. `.incidence()` integrates each
save interval (trapezoid by default) to a count; times become the interval
right edges.

In [ ]:
weekly = (
    res["infection"]
    .incidence()
    .sum_over(age, side="dest")
    .to_frame()
)
assert "age=0-4" in weekly.columns
assert weekly.height == 28  # T-1 intervals from 29 save points (steps+1)
print(weekly.head())


## Rate vs incidence: the quadrature gap

Trapezoid over save intervals is second-order accurate; it is **not** the
solver's own accumulated incidence. The difference is visible and quantified
below — save finer (or use `method="simpson"`) when calibrating to case counts.

In [ ]:
# Analytic stand-in: integrate f(t)=t^2 on [0, 1].
from summer4.results.trace import Trace
from summer4.time import TimeAxis

ts = np.linspace(0.0, 1.0, 5)
rate = Trace(
    times=TimeAxis(values=ts, epoch=None, kind="explicit"),
    values=ts**2,
    dims=("time",),
)
analytic = 1.0 / 3.0
trap = float(np.asarray(rate.integrate(method="trapezoid").values))
simp = float(np.asarray(rate.integrate(method="simpson").values))
print(f"analytic={analytic:.6f}  trapezoid={trap:.6f}  err={abs(trap - analytic):.2e}")
print(f"analytic={analytic:.6f}  simpson={simp:.6f}    err={abs(simp - analytic):.2e}")
assert abs(simp - analytic) < abs(trap - analytic)


## Kleene-unknown on an exit flow

`Dest(Everything())` on a death flow selects nothing (the destination side is
absent). The same selector on a transition flow keeps every edge.

In [ ]:
death_dest = res["death"].select(Dest(Everything()))
inf_dest = res["infection"].select(Dest(Everything()))
assert death_dest.values.pmap.size == 0
assert inf_dest.values.pmap.size == res["infection"].values.pmap.size

# Only edges that actually change age:
moved = res["ageing"].select(cm.edges("ageing").moves_mask(age))
assert moved.values.pmap.size == res["ageing"].values.pmap.size
print("exit Dest(Everything()) edges:", death_dest.values.pmap.size)
print("infection Dest(Everything()) edges:", inf_dest.values.pmap.size)
